# Modern Application Development – I: Comprehensive Lecture Notes  
**Week 3: Views, UI Design, Tools, Accessibility & Practical Templating**

---

## Table of Contents
1. [Overview of the MVC Architecture (Recap)](#1-overview-of-the-mvc-architecture)
2. [Deep Dive into Views](#2-deep-dive-into-views)
3. [User Interface Design Principles](#3-user-interface-design-principles)
4. [Usability Heuristics (Jakob Nielsen’s 10 Heuristics)](#4-usability-heuristics)
5. [Tools for View Design – Wireframing](#5-tools-for-view-design--wireframing)
6. [Tools for View Implementation – Programmatic HTML and Templates](#6-tools-for-view-implementation--programmatic-html-and-templates)
7. [Jinja2 Templating in Python (Screencasts)](#7-jinja2-templating-in-python)
8. [Accessibility in Web Applications](#8-accessibility-in-web-applications)
9. [Command Line Arguments in Python](#9-command-line-arguments-in-python)
10. [Browser Developer Tools](#10-browser-developer-tools)

---

## 1. Overview of the MVC Architecture (Recap)

The Model‑View‑Controller (MVC) paradigm is a **software architectural pattern** that separates an application into three interconnected components. Its goal is to isolate **business logic** from **user interface** concerns, making the application easier to manage, test, and evolve.

### 1.1 Historical Roots
MVC was introduced in the late 1970s as part of the **Smalltalk‑80** programming environment developed at Xerox PARC. Smalltalk was a pioneering object‑oriented language, and MVC leveraged its message‑passing nature. Since then, MVC has become one of the most influential patterns in graphical user interface (GUI) and web application design. While the original formulation has been adapted and sometimes criticised, the core idea of separating data, presentation, and control remains a cornerstone of software engineering.

### 1.2 The Three Components

**Model**
- Represents the **data** and **business rules** of the application.
- It is independent of the user interface; it does not know how the data will be displayed.
- In an email client, the Model stores emails, their metadata (sender, date, subject, read/unread status), folder hierarchies, and the rules for spam filtering.
- In our running example (a student grade book), the Model includes tables for **students** (ID, name), **courses** (ID, name), and **marks** (student ID, course ID, score). It also enforces constraints like “a student cannot have two marks for the same course”.

**View**
- The **presentation layer** – anything the user sees or interacts with.
- It renders the Model’s data in a human‑friendly format and captures user input.
- For the grade book, possible Views include:
  - A table showing all marks for a given student.
  - A histogram of marks distribution for a course.
  - A JSON document providing raw data to another application.
- The View never modifies the Model directly; it only reflects the Model’s current state.

**Controller**
- The **glue** between Model and View.
- It receives user input from the View, translates it into actions on the Model (e.g., “add a new student”, “update a mark”), and may select a new View to present.
- In a web application, the Controller is typically the server‑side code that handles HTTP requests. For example, when a user submits a form with new marks, the Controller validates the data, updates the Model, and redirects to the updated marks list View.

### 1.3 MVC as a Composition of Design Patterns
MVC is not a single design pattern but a combination of several:
- **Observer pattern**: The View observes the Model for changes (or the Controller notifies the View after modifying the Model). When the Model changes, the View automatically updates.
- **Strategy pattern**: The Controller defines the behaviour that connects user actions to Model updates. Different Controllers can be plugged in to change how input is interpreted.
- **Composite pattern**: The View is often composed of nested sub‑views (a complex page may contain a header, sidebar, main content, footer, each with its own sub‑view).

This separation enables **parallel development**: a UI designer can work on the View (HTML/CSS), a database specialist on the Model, and a backend developer on the Controller, all with minimal interference.

### 1.4 The Student Grade Book Example
To anchor the discussion, the lecture introduces a concrete example that will be used throughout the course:
- **Students**: identified by an ID and a name.
- **Courses**: identified by an ID and a name.
- **Marks**: a relationship linking a student ID and a course ID to a numeric score.
- The underlying data can be thought of as a spreadsheet or a set of database tables.
- Multiple Views are possible:
  - **Student‑centric view**: enter a student ID, see their name and all their course marks.
  - **Course‑centric view**: enter a course ID, see the list of enrolled students with marks, statistics (mean, histogram).
- Controllers will handle adding/updating students, courses, and marks.

This example is intentionally simple but realistic – it forces us to confront data validation, relationships, and multiple output formats, all of which are central to web application development.

---

## 2. Deep Dive into Views

A **View** is any output from a program that is presented to an external entity. That entity may be a human (via a screen, speaker, Braille display) or another machine (via JSON, XML). In the context of web applications, we focus primarily on **HTML pages** served to a browser, but we must keep the broader definition in mind.

### 2.1 The Two Sides of a View
Every View has two aspects:
1. **User Interface (UI)** – what the user perceives: visual layout, colours, fonts, sounds, haptic feedback. This is the *output* channel.
2. **User Interaction** – how the user provides input: keyboard, mouse, touch, voice, gestures, even physical handles on a door.

A well‑designed View harmonises these two sides. The UI must present information clearly; the interaction must be intuitive and efficient.

### 2.2 Hardware Constraints and User Agents
The actual interaction is heavily determined by the **client device**:
- A desktop PC has a large screen, a full keyboard, and a precise mouse.
- A smartphone has a small touchscreen, an on‑screen keyboard, and sensors (accelerometer, GPS).
- A smart speaker has only audio output and voice input.
- An embedded device may have just a single button and an LED.

The web server can often infer the client’s capabilities from the **User‑Agent** string and other HTTP headers. Responsive design (using CSS media queries) and progressive enhancement are strategies to serve an acceptable experience across all devices. The lecture emphasises that **as a designer, you rarely control the client’s hardware**; you must design for diversity.

### 2.3 Types of Views by Dynamism
**Fully Static Pages**
- The HTML file is stored on the server and sent verbatim for every request.
- Example: a simple “About Us” page that rarely changes.
- Even apparently static pages like Google’s homepage are, in reality, highly dynamic – they personalise based on login state, location, etc.

**Partly Dynamic Pages**
- Some parts are static, others are generated on the fly.
- Example: Wikipedia’s main page, which has static sections (logo, navigation) alongside dynamic sections (“Today’s featured article”, “In the news”). The server assembles the page from a template and fills in the dynamic bits from a database or external source.

**Mostly Dynamic Pages**
- Almost the entire page is generated per request, often personalised.
- Example: Amazon’s homepage, which shows recommended products, deals specific to the user’s browsing history, location‑based offers, etc. Only the core layout and static assets are reused; the content is entirely computed.

### 2.4 Output Formats – HTML is not the Only View
While HTML is the most common, a View can be:
- **Rendered HTML**: sent to a browser for direct display. This is what we normally think of as a web page.
- **Images**: dynamically generated graphs, charts, or maps (e.g., a PNG of a marks histogram). The server creates the image and the browser embeds it in an `<img>` tag.
- **Structured data**: JSON or XML. These are machine‑readable formats. They are not meant for direct human consumption but for consumption by other software (e.g., a mobile app that fetches data from the same server that powers the website, or a third‑party integration). This is the foundation of a **REST API**.
- **Plain text**: sometimes the simplest possible view is just raw text, useful for logging, debugging, or feeding into command‑line tools.

The choice of output format depends entirely on the **consumer** of the View. The Model and Controller remain unchanged; only the View layer varies.

---

## 3. User Interface Design Principles

User Interface Design is a discipline that blends psychology, design, and engineering. While aesthetics are subjective, there are widely agreed‑upon principles that make an interface **simple** and **efficient**.

### 3.1 Core Goals: Simplicity and Efficiency
- **Simplicity**: the user should immediately understand what the interface does and how to use it. The classic “push” bar on a door is self‑explanatory; you press it and the door opens.
- **Efficiency**: the user achieves their goal with minimal effort. Keyboard shortcuts in Gmail (e.g., `c` to compose, `e` to archive) allow expert users to navigate without moving their hands to the mouse.

These two goals often reinforce each other, but they can conflict. A voice‑controlled door might be simple (“just say ‘open’”), but it’s slower and less reliable than a push bar for most people.

### 3.2 Aesthetics vs. Accessibility
- **Aesthetics**: how good something looks. It is subjective but heavily influenced by cultural trends, colour theory, and principles of visual hierarchy. The evolution of iOS icons from glossy, gradient‑filled designs to flat, minimalist icons illustrates changing aesthetic preferences. The lecture advises beginners to **rely on established design systems** (like Bootstrap, Material Design) rather than inventing their own colour schemes and layouts, because these systems have been refined by professional designers and tested on millions of users.
- **Accessibility**: how usable the interface is for people with disabilities (visual, auditory, motor, cognitive). There is often a tension: high‑contrast, large‑font designs needed for low‑vision users may look “loud” to fully‑sighted users. The principle is that **accessibility is not optional**; it is a requirement for ethical, inclusive, and often legally compliant software.

### 3.3 The Systematic Process of UI Design
Building a good UI is not guesswork. It follows an engineering process:

1. **Functionality Requirements Gathering**: Talk to stakeholders (the client, the actual users) to understand what the application must do. For a gradebook, the academic section might need:
   - Bulk import of students from a CSV file.
   - Manual entry of individual marks.
   - A way to lock results after finalisation.
   - A dashboard showing failure rates per course.

2. **User and Task Analysis**: Identify who will use the system and what tasks they will perform. The administrator has different needs than a student viewing their own grades. Task analysis breaks down each task into steps, revealing where the UI must provide guidance or shortcuts.

3. **Prototyping**: Create low‑fidelity **wireframes** or **mock‑ups**. These show layout, navigation, and key elements without full functionality. They are cheap to produce and iterate. Feedback from users at this stage prevents costly rework later.

4. **Implementation**: Translate the approved prototype into actual code (HTML, CSS, server‑side logic).

5. **User Acceptance Testing**: Real users perform tasks using the implemented application. Observations and feedback may lead to further refinements. This cycle often repeats (agile development).

---

## 4. Usability Heuristics (Jakob Nielsen’s 10 Heuristics)

In 1994, Jakob Nielsen published a set of **ten general principles** for interaction design. They are called *heuristics* because they are rules of thumb, not rigid laws. They apply to virtually any user interface, from a website to a microwave oven.

1. **Visibility of system status**  
   The system should always keep users informed about what is going on, through appropriate feedback within reasonable time.  
   *Example*: The mouse cursor changes to a hand over a hyperlink, indicating it is clickable. A progress bar shows how much of a file upload remains.

2. **Match between system and the real world**  
   The system should speak the user’s language, with words, phrases and concepts familiar to the user, rather than system‑oriented terms. Follow real‑world conventions, making information appear in a natural and logical order.  
   *Example*: On a stove, the knob layout mirrors the burner layout. In a shopping cart, the “checkout” process mimics a physical store.

3. **User control and freedom**  
   Users often choose system functions by mistake and need a clearly marked “emergency exit” to leave the unwanted state without having to go through an extended dialogue. Support undo and redo.  
   *Example*: “Undo Send” in Gmail; the recycle bin on desktops.

4. **Consistency and standards**  
   Users should not have to wonder whether different words, situations, or actions mean the same thing. Follow platform and industry conventions.  
   *Example*: Links are always underlined and blue by default; changing this breaks user expectations. All “Save” buttons across an application should look and behave identically.

5. **Error prevention**  
   Even better than good error messages is a careful design which prevents a problem from occurring in the first place.  
   *Example*: Drop‑down menus instead of free‑text fields for selecting a country; they limit the input to valid options. Confirmation dialogs before deleting an account.

6. **Recognition rather than recall**  
   Minimise the user’s memory load by making objects, actions, and options visible. The user should not have to remember information from one part of the dialogue to another.  
   *Example*: A toolbar with icons and tooltips is easier than requiring users to memorise keyboard shortcuts (though shortcuts can be provided for experts – see #7).

7. **Flexibility and efficiency of use**  
   Accelerators – unseen by the novice user – may often speed up the interaction for the expert user such that the system can cater to both inexperienced and experienced users. Allow users to tailor frequent actions.  
   *Example*: Keyboard shortcuts (Ctrl+S, Ctrl+Z). Macros in Excel.

8. **Aesthetic and minimalist design**  
   Dialogues should not contain information which is irrelevant or rarely needed. Every extra unit of information in a dialogue competes with the relevant units of information and diminishes their relative visibility.  
   *Example*: Google’s homepage, which consists of a logo, a search box, and two buttons. Cluttered interfaces slow down task completion.

9. **Help users recognise, diagnose, and recover from errors**  
   Error messages should be expressed in plain language (no codes), precisely indicate the problem, and constructively suggest a solution.  
   *Example*: Instead of “Error 404”, a page saying “We couldn’t find that page. Here are some links that might help.”

10. **Help and documentation**  
    Even though it is better if the system can be used without documentation, it may be necessary to provide help and documentation. Any such information should be easy to search, focused on the user’s task, list concrete steps to be carried out, and not be too large.  
    *Example*: Context‑sensitive help: clicking a “?” icon next to a field explains what that field does.

These heuristics are not step‑by‑step instructions but lenses through which to evaluate a design. The lecture strongly encourages reading the full Nielsen Norman Group articles for a deeper understanding.

---

## 5. Tools for View Design – Wireframing

Before writing any code, it is best to sketch the structure of the user interface. A **wireframe** is a visual guide that represents the skeletal framework of a page. It shows:

- **Information hierarchy**: what content is most important, where it is placed.
- **Navigation**: how users move between pages or sections.
- **Layout**: the arrangement of blocks (header, footer, sidebar, main content area).
- **Placeholder content**: typically “lorem ipsum” text, grey boxes for images.

Wireframes are deliberately devoid of colour, typography, and detailed graphics. Their purpose is to focus on **structure and functionality**, not final aesthetics.

**Example from the lecture**: A wireframe for a user directory profile might show a large placeholder for the user’s photo at the top, the name directly beneath, then contact details (email, phone), a “categories” tag cloud, a short description, and some thumbnails for attachments. The wireframe helps the client and developer agree on the essential elements before any design work.

**Tools for wireframing**:
- **Pen and paper**: the fastest, cheapest way. Use graph paper to keep proportions.
- **Dedicated software**: Lucidchart, Balsamiq, Sketch, Figma, Adobe XD. Many offer free tiers for students.
- The process is iterative: start with a low‑fidelity sketch, present it to stakeholders, incorporate feedback, then gradually increase fidelity. A wireframe can evolve into a **mock‑up** (adding colours and fonts) and finally a **prototype** (clickable with limited functionality).

---

## 6. Tools for View Implementation – Programmatic HTML and Templates

Once the wireframe is approved, we must turn it into actual HTML. There are several approaches, each with trade‑offs.

### 6.1 Direct Print Statements
The most basic method: use `print()` to output HTML strings.
```python
print("<html><head><title>My Page</title></head>")
print("<body><h1>Welcome</h1></body></html>")
```
This works for tiny examples but is unmaintainable for real applications:
- It is error‑prone (forgotten closing tags, mismatched quotes).
- The code is cluttered, making it hard to see the page structure.
- Mixing Python logic with HTML makes both harder to read.

### 6.2 Helper Functions for HTML Tags
A slight improvement: define functions that generate tags.
```python
def h1(text):
    return f"<h1>{text}</h1>"
```
This reduces typos and can include attribute handling. Libraries like **pyhtml** take this approach to the extreme, allowing us to write:
```python
from pyhtml import *
t = html(
    head(title("Test Page")),
    body(
        h1("Hello"),
        p("This is a paragraph.")
    )
)
print(t.render())
```
The output is valid, indented HTML. Because it’s pure Python, we can use loops, conditionals, and list comprehensions to build complex tables or lists. However, the resulting code is still a tree of function calls that mirrors the HTML structure. If a designer later wants to change the layout, they must wade through Python code. This violates the separation of concerns between development and design.

### 6.3 Templating Engines
A **template** is a text file that contains the static parts of the output (the “skeleton”) with special placeholders for dynamic data. The application code reads the template, provides the data, and the template engine produces the final output. This cleanly separates the **presentation logic** (in the template) from the **business logic** (in the Python code).

**Simple string templates (Python’s `string.Template`)**:
```python
from string import Template
t = Template("$name is the $job of $company.")
print(t.substitute(name="Tim Cook", job="CEO", company="Apple Inc."))
```
This is good for small substitutions but lacks features like loops and conditionals.

**Jinja2** (the template engine used with Flask):
- Uses `{{ variable }}` for expression substitution.
- Uses `{% statement %}` for control flow (loops, conditionals).
- Supports template inheritance (a base template with blocks that child templates can override).
- Auto‑escapes HTML to prevent XSS attacks by default.

A Jinja2 template for a student marks table might look like:
```html
<table>
  <tr><th>Course</th><th>Marks</th></tr>
  {% for row in marks %}
  <tr><td>{{ row.course }}</td><td>{{ row.score }}</td></tr>
  {% endfor %}
</table>
```
The Python code is minimal: load the template, call `render(marks=data)`, and return the result. The template is readable by front‑end developers who may not know Python.

The lecture emphasises that **Jinja2 can generate any text format**, not just HTML. You could generate JSON, CSV, email bodies, or even Python source code. This flexibility makes templating a universal tool in a developer’s toolkit.

---

## 7. Jinja2 Templating in Python (Screencasts)

The screencasts provide a hands‑on walkthrough of using Jinja2 to generate HTML dynamically from Python data.

### 7.1 Setup and Virtual Environment
- Create a project directory.
- Set up a **Python virtual environment** (`python3 -m venv env`). This isolates project dependencies from the system Python, preventing version conflicts.
- Activate it (`source env/bin/activate`).
- Install Jinja2 with `pip install jinja2`. The `requirements.txt` file (generated via `pip freeze > requirements.txt`) lists all dependencies so others can replicate the environment.

### 7.2 Basic Jinja2 Usage
```python
from jinja2 import Template
t = Template("Hello {{ name }}!")
print(t.render(name="Thej"))
```
Key points:
- `Template` is a class that compiles the template string into an internal representation.
- `render()` takes keyword arguments corresponding to the variables used in the template.
- The double curly braces `{{ ... }}` denote an **expression** that will be evaluated and inserted into the output.

### 7.3 Control Structures: Loops and Conditionals
Jinja2 supports `{% for %}` and `{% if %}`:
```html
<ul>
{% for item in items %}
  <li>{{ item.name }}</li>
{% endfor %}
</ul>
```
Unlike Python, Jinja2 requires an explicit `{% endfor %}` because indentation is not used to denote blocks (templates are a mix of HTML and logic). Conditionals look similar:
```html
{% if user.is_admin %}
  <a href="/admin">Admin Panel</a>
{% endif %}
```

### 7.4 Separating Template from Code
Instead of embedding the template as a string in Python, it is best to keep it in a separate file, e.g., `template.html.jinja2`. The Python code then reads the file:
```python
with open('template.html.jinja2') as file:
    template_str = file.read()
t = Template(template_str)
```
This separation means that a designer can edit the template with an HTML editor, while the developer works on the Python logic.

### 7.5 Rendering a Complete HTML Table
The example takes a list of dictionaries (Jnanpith awardees) and generates an HTML table:
- The template defines the `<table>`, `<thead>`, and a `<tbody>` with a `{% for %}` loop over the data.
- The Python data is a list of dicts: `[{'year': 1965, 'awardee': 'G. Sankara Kurup', 'language': 'Malayalam'}, ...]`.
- In the template, each row accesses dictionary keys using Python‑like syntax: `{{ awardee['year'] }}` or `{{ awardee.year }}`.
- The rendered HTML is written to a file (`janapith.html`) which can then be opened in a browser.

### 7.6 Advantages of Jinja2
- **Conciseness**: A few lines of template can replace dozens of `print` statements.
- **Readability**: The template looks almost like the final HTML, with minimal logic intruding.
- **Extensibility**: Jinja2 supports macros (reusable snippets), filters (e.g., `{{ name|title }}`), and template inheritance (a “base” template with a common layout, and “child” templates that fill in the content blocks).
- **Security**: By default, Jinja2 escapes HTML special characters in substituted variables, thwarting Cross‑Site Scripting (XSS) attacks.

The screencast encourages further exploration of Jinja2’s documentation for advanced features like template inheritance, which is essential for maintaining a consistent look across many pages of an application.

---

## 8. Accessibility in Web Applications

Accessibility (often abbreviated as **a11y**) means designing web content so that people with disabilities can perceive, understand, navigate, and interact with it. It is a fundamental aspect of ethical software development and is mandated by laws in many countries.

### 8.1 Why Accessibility Matters
- **Moral imperative**: The web should be open to everyone, regardless of ability.
- **Legal requirements**: Many governments require public websites to meet accessibility standards (e.g., Section 508 in the US, EN 301 549 in the EU).
- **Business case**: Accessible sites reach a larger audience, rank better in search engines (semantic HTML aids SEO), and generally offer a better user experience for *all* users (e.g., captions help in noisy environments).
- **Types of disabilities** include:
  - **Visual**: blindness, low vision, colour blindness.
  - **Auditory**: deafness, hard of hearing.
  - **Motor**: inability to use a mouse, slow response time, limited fine motor control.
  - **Cognitive**: learning disabilities, distractibility, difficulty reading.

### 8.2 WCAG – Web Content Accessibility Guidelines
The W3C publishes the **Web Content Accessibility Guidelines (WCAG)**. The current version (2.1) is organised around four principles, often remembered by the acronym **POUR**:

1. **Perceivable** – Information and user interface components must be presentable to users in ways they can perceive.
   - Provide **text alternatives** for non‑text content (`alt` attributes for images, captions for videos).
   - Content must be adaptable and distinguishable (e.g., sufficient colour contrast, resizable text without loss of content).

2. **Operable** – User interface components and navigation must be operable.
   - All functionality must be available from a **keyboard** (no mouse‑only interactions).
   - Users must have enough time to read and use content (no auto‑advancing carousels without pause controls).
   - Do not design content in a way that is known to cause **seizures** (no flashing content > 3 times per second).
   - Provide ways to help users navigate, find content, and determine where they are (skip‑to‑main‑content links, meaningful page titles).

3. **Understandable** – Information and the operation of the user interface must be understandable.
   - Text content must be readable and understandable (use plain language).
   - Web pages must appear and operate in predictable ways (consistent navigation, no unexpected pop‑ups).
   - Help users avoid and correct mistakes (input validation, clear error messages).

4. **Robust** – Content must be robust enough that it can be interpreted reliably by a wide variety of user agents, including assistive technologies.
   - Use valid, semantic HTML so that screen readers and other tools can parse the document correctly.
   - ARIA (Accessible Rich Internet Applications) attributes can enhance accessibility of dynamic content, but should be used as a supplement, not a substitute for proper HTML.

### 8.3 Practical Accessibility Techniques
- **Semantic HTML**: Use `<header>`, `<nav>`, `<main>`, `<article>`, `<aside>`, `<footer>` instead of generic `<div>`s. This provides a logical structure that screen readers can navigate.
- **Form labels**: Every form input should have a `<label>` associated with it, so screen readers announce what each field is for.
- **Focus management**: Ensure that keyboard focus order follows the visual order. Do not trap focus in a modal without a way to close it via keyboard.
- **Colour and contrast**: Ensure a contrast ratio of at least 4.5:1 for normal text. Do not rely on colour alone to convey information (e.g., add an icon next to error messages, not just a red border).
- **Testing**: Use automated tools (WAVE, axe, Lighthouse) and manual testing (keyboard‑only navigation, screen reader testing with NVDA or VoiceOver). The W3C provides a full suite of techniques for meeting each WCAG success criterion.

The lecture strongly stresses that **accessibility is an integral part of development, not an afterthought**. Designing with accessibility in mind from the beginning is far cheaper and results in a better product than retrofitting later.

---

## 9. Command Line Arguments in Python

This screencast covers a fundamental skill: executing Python scripts from the terminal and passing parameters to them. This is crucial for building server‑side scripts that can be configured without modifying source code.

### 9.1 Running a Python Script from the Terminal
Normally, an IDE provides a “Run” button. On a server or in an automated pipeline, we use the command line:
```bash
python script.py
```
(On some systems, `python3` is required to avoid invoking Python 2.)

If the script contains only `print("Hello")`, that output appears in the terminal. The script can also use `input()` to accept user input interactively.

### 9.2 Passing Command Line Arguments
Arguments are written after the script name, separated by spaces:
```bash
python hello.py 123 Abhishek
```
These arguments are accessible inside the script via the **`sys.argv`** list, provided by the built‑in `sys` module.

### 9.3 The `sys.argv` List
- `sys.argv` is a Python list of strings.
- `sys.argv[0]` is **always the script name** (e.g., `hello.py`).
- `sys.argv[1]` is the first argument, `sys.argv[2]` the second, and so on.
- The length `len(sys.argv)` includes the script name, so if you pass two arguments, the length is 3.

Example:
```python
import sys
print(f"Total parameters: {len(sys.argv)}")
print(f"First parameter: {sys.argv[1]}")
print(f"Second parameter: {sys.argv[2]}")
```
Running `python hello.py 123 Abhishek` prints:
```
Total parameters: 3
First parameter: 123
Second parameter: Abhishek
```

### 9.4 Important Notes
- **All arguments are strings**, even if they look like numbers. You must explicitly convert them using `int()`, `float()`, etc., if you need numerical values.
- You can pass any number of arguments; the script can ignore excess ones or use only what it needs.
- Command line arguments are a simple way to parameterise scripts (e.g., specify a port number, a filename, or a logging level). For more complex option parsing, Python’s `argparse` module is recommended, but `sys.argv` suffices for basic cases.

This technique is the foundation for writing server‑side utilities, background jobs, and even for the first steps towards building an HTTP server that responds to different paths.

---

## 10. Browser Developer Tools

Modern browsers are not just for viewing web pages; they include a powerful suite of **developer tools** (DevTools) for inspecting, debugging, and profiling web applications. The screencast explores the most essential panels using Chrome as an example, but Firefox’s tools are nearly identical.

### 10.1 Opening DevTools
- Right‑click on any element and select **Inspect**, or press `F12` (or `Ctrl+Shift+I`).
- The DevTools pane can be docked to the bottom, side, or popped out as a separate window.

### 10.2 The Elements (Inspector) Panel
- Shows the live DOM tree of the current page. As you hover over nodes, the corresponding area on the page is highlighted.
- You can edit the HTML directly: double‑click a tag, modify text, or add/delete elements. These changes are temporary and vanish on refresh.
- The **Styles** sub‑panel on the right shows all CSS rules that apply to the selected element, grouped by selector. The **Computed** sub‑panel shows the final resolved values.
- You can toggle individual rules on/off, edit property values, or add new styles on the fly. This is invaluable for experimenting with design without touching source code.
- The cascade is visible: overridden properties are shown with a strikethrough, and clicking the arrow next to a property reveals its origin.

### 10.3 The Console Panel
- A JavaScript REPL (Read‑Eval‑Print Loop). You can type arbitrary JavaScript and see the results immediately.
- It is the primary output destination for `console.log()`, `console.error()`, `console.warn()`, etc.
- Useful for debugging: instead of intrusive `alert()` calls, developers log messages, objects, and variables to the console.
- `console.table(data)` prints an array or object as a formatted table, making it easier to inspect structured data.
- `console.time("label")` and `console.timeEnd("label")` measure the execution time of code blocks.
- `console.clear()` wipes the console.

### 10.4 The Sources Panel
- Shows all files that make up the current page: HTML, CSS, JavaScript, images, fonts.
- For JavaScript, you can **set breakpoints** by clicking on line numbers. When execution reaches that line, the debugger pauses, allowing you to inspect variables, step through code, and watch the call stack.
- This is the same debugging paradigm found in IDE debuggers and is essential for understanding complex client‑side logic.

### 10.5 The Network Panel
- Records every HTTP request the page makes (HTML, CSS, JS, images, API calls).
- For each request, it shows:
  - **Status code**, type (`GET`, `POST`), size, and **time**.
  - **Headers** (both request and response) – useful for checking cookies, content types, caching directives.
  - **Preview/Response** – see the actual data returned.
- The **waterfall** visualisation breaks down the timing of each request (queuing, DNS lookup, TCP connection, SSL handshake, waiting for server, downloading content). This helps identify bottlenecks.
- You can filter requests by type (XHR for API calls, JS, CSS, etc.).
- **Preserve log** keeps requests visible across page navigations.
- **Throttling** allows simulating slow network conditions (e.g., Slow 3G) to test performance on mobile.

### 10.6 The Application Panel
- Inspects client‑side storage:
  - **Local Storage** and **Session Storage**: key‑value stores accessible via JavaScript. You can view, edit, and delete entries.
  - **Cookies**: see all cookies set by the page, their domains, paths, expiry dates, and flags (`HttpOnly`, `Secure`).
  - **Cache Storage**: view cached responses (e.g., service worker caches).
  - **IndexedDB**: a low‑level API for storing large amounts of structured data. The panel allows browsing databases, object stores, and records.

### 10.7 Other Panels
- **Performance**: record and analyse runtime performance (CPU usage, rendering, scripting).
- **Memory**: profile memory usage and find leaks.
- **Lighthouse**: an integrated audit tool that scores the page on performance, accessibility, best practices, and SEO.

The screencast emphasises that **the browser is the developer’s most important tool**. Learning DevTools thoroughly pays enormous dividends in debugging productivity. The examples with `httpbin` – a service for testing HTTP requests – show how to examine POST form data, verify response headers, and understand the whole request‑response cycle.

---

## Summary of Week 3

- **MVC** provides a proven structure: Model (data), View (presentation), Controller (logic). This separation is the backbone of the course’s approach to app development.
- **Views** are any output presented to a user or another machine. They must adapt to diverse hardware, can be static or dynamic, and may produce HTML, images, JSON, or plain text.
- **UI Design** aims for simplicity and efficiency. It follows a systematic process of requirements, prototyping, and testing. Established heuristics (Nielsen’s 10) guide designers towards usable products.
- **Wireframes** are low‑fidelity blueprints that define layout and navigation before coding begins.
- **HTML generation** can be done programmatically (using helper functions or libraries like pyhtml) but **templates** (especially Jinja2) offer a superior separation of concerns. Jinja2 is a powerful, extensible templating engine that supports variables, loops, conditionals, and inheritance.
- **Accessibility** is a core requirement, not a nice‑to‑have. The WCAG’s POUR principles (Perceivable, Operable, Understandable, Robust) provide a comprehensive framework. Semantic HTML, keyboard navigation, contrast, and text alternatives are key techniques.
- **Command line arguments** enable scripts to accept parameters at runtime via `sys.argv`, a crucial skill for backend development.
- **Browser DevTools** are the developer’s Swiss Army knife: Elements for HTML/CSS debugging, Console for JavaScript logging, Sources for debugging, Network for request inspection, and Application for storage inspection.

This week moves from abstract design principles to concrete tools and techniques, equipping students to begin building the **View** layer of their web applications with modern, maintainable, and inclusive practices.

## Table of Contents
1. [Overview of the MVC Architecture (Recap)](#1-overview-of-the-mvc-architecture)
2. [Deep Dive into Views](#2-deep-dive-into-views)
3. [User Interface Design Principles](#3-user-interface-design-principles)
4. [Usability Heuristics (Jakob Nielsen’s 10 Heuristics)](#4-usability-heuristics)
5. [Tools for View Design – Wireframing](#5-tools-for-view-design--wireframing)
6. [Tools for View Implementation – Programmatic HTML and Templates](#6-tools-for-view-implementation--programmatic-html-and-templates)
7. [Jinja2 Templating in Python (Screencasts)](#7-jinja2-templating-in-python)
8. [Accessibility in Web Applications](#8-accessibility-in-web-applications)
9. [Command Line Arguments in Python](#9-command-line-arguments-in-python)
10. [Browser Developer Tools](#10-browser-developer-tools)

---

## 1. Overview of the MVC Architecture (Recap)

The Model‑View‑Controller (MVC) paradigm is a **software architectural pattern** that separates an application into three interconnected components. Its goal is to isolate **business logic** from **user interface** concerns, making the application easier to manage, test, and evolve.

### 1.1 Historical Roots
MVC was introduced in the late 1970s as part of the **Smalltalk‑80** programming environment developed at Xerox PARC. Smalltalk was a pioneering object‑oriented language, and MVC leveraged its message‑passing nature. Since then, MVC has become one of the most influential patterns in graphical user interface (GUI) and web application design. While the original formulation has been adapted and sometimes criticised, the core idea of separating data, presentation, and control remains a cornerstone of software engineering.

### 1.2 The Three Components

**Model**
- Represents the **data** and **business rules** of the application.
- It is independent of the user interface; it does not know how the data will be displayed.
- In an email client, the Model stores emails, their metadata (sender, date, subject, read/unread status), folder hierarchies, and the rules for spam filtering.
- In our running example (a student grade book), the Model includes tables for **students** (ID, name), **courses** (ID, name), and **marks** (student ID, course ID, score). It also enforces constraints like “a student cannot have two marks for the same course”.

**View**
- The **presentation layer** – anything the user sees or interacts with.
- It renders the Model’s data in a human‑friendly format and captures user input.
- For the grade book, possible Views include:
  - A table showing all marks for a given student.
  - A histogram of marks distribution for a course.
  - A JSON document providing raw data to another application.
- The View never modifies the Model directly; it only reflects the Model’s current state.

**Controller**
- The **glue** between Model and View.
- It receives user input from the View, translates it into actions on the Model (e.g., “add a new student”, “update a mark”), and may select a new View to present.
- In a web application, the Controller is typically the server‑side code that handles HTTP requests. For example, when a user submits a form with new marks, the Controller validates the data, updates the Model, and redirects to the updated marks list View.

### 1.3 MVC as a Composition of Design Patterns
MVC is not a single design pattern but a combination of several:
- **Observer pattern**: The View observes the Model for changes (or the Controller notifies the View after modifying the Model). When the Model changes, the View automatically updates.
- **Strategy pattern**: The Controller defines the behaviour that connects user actions to Model updates. Different Controllers can be plugged in to change how input is interpreted.
- **Composite pattern**: The View is often composed of nested sub‑views (a complex page may contain a header, sidebar, main content, footer, each with its own sub‑view).

This separation enables **parallel development**: a UI designer can work on the View (HTML/CSS), a database specialist on the Model, and a backend developer on the Controller, all with minimal interference.

### 1.4 The Student Grade Book Example
To anchor the discussion, the lecture introduces a concrete example that will be used throughout the course:
- **Students**: identified by an ID and a name.
- **Courses**: identified by an ID and a name.
- **Marks**: a relationship linking a student ID and a course ID to a numeric score.
- The underlying data can be thought of as a spreadsheet or a set of database tables.
- Multiple Views are possible:
  - **Student‑centric view**: enter a student ID, see their name and all their course marks.
  - **Course‑centric view**: enter a course ID, see the list of enrolled students with marks, statistics (mean, histogram).
- Controllers will handle adding/updating students, courses, and marks.

This example is intentionally simple but realistic – it forces us to confront data validation, relationships, and multiple output formats, all of which are central to web application development.

---


## 2. Deep Dive into Views

A **View** is any output from a program that is presented to an external entity. That entity may be a human (via a screen, speaker, Braille display) or another machine (via JSON, XML). In the context of web applications, we focus primarily on **HTML pages** served to a browser, but we must keep the broader definition in mind.

### 2.1 The Two Sides of a View
Every View has two aspects:
1. **User Interface (UI)** – what the user perceives: visual layout, colours, fonts, sounds, haptic feedback. This is the *output* channel.
2. **User Interaction** – how the user provides input: keyboard, mouse, touch, voice, gestures, even physical handles on a door.

A well‑designed View harmonises these two sides. The UI must present information clearly; the interaction must be intuitive and efficient.

### 2.2 Hardware Constraints and User Agents
The actual interaction is heavily determined by the **client device**:
- A desktop PC has a large screen, a full keyboard, and a precise mouse.
- A smartphone has a small touchscreen, an on‑screen keyboard, and sensors (accelerometer, GPS).
- A smart speaker has only audio output and voice input.
- An embedded device may have just a single button and an LED.

The web server can often infer the client’s capabilities from the **User‑Agent** string and other HTTP headers. Responsive design (using CSS media queries) and progressive enhancement are strategies to serve an acceptable experience across all devices. The lecture emphasises that **as a designer, you rarely control the client’s hardware**; you must design for diversity.

### 2.3 Types of Views by Dynamism
**Fully Static Pages**
- The HTML file is stored on the server and sent verbatim for every request.
- Example: a simple “About Us” page that rarely changes.
- Even apparently static pages like Google’s homepage are, in reality, highly dynamic – they personalise based on login state, location, etc.

**Partly Dynamic Pages**
- Some parts are static, others are generated on the fly.
- Example: Wikipedia’s main page, which has static sections (logo, navigation) alongside dynamic sections (“Today’s featured article”, “In the news”). The server assembles the page from a template and fills in the dynamic bits from a database or external source.

**Mostly Dynamic Pages**
- Almost the entire page is generated per request, often personalised.
- Example: Amazon’s homepage, which shows recommended products, deals specific to the user’s browsing history, location‑based offers, etc. Only the core layout and static assets are reused; the content is entirely computed.

### 2.4 Output Formats – HTML is not the Only View
While HTML is the most common, a View can be:
- **Rendered HTML**: sent to a browser for direct display. This is what we normally think of as a web page.
- **Images**: dynamically generated graphs, charts, or maps (e.g., a PNG of a marks histogram). The server creates the image and the browser embeds it in an `<img>` tag.
- **Structured data**: JSON or XML. These are machine‑readable formats. They are not meant for direct human consumption but for consumption by other software (e.g., a mobile app that fetches data from the same server that powers the website, or a third‑party integration). This is the foundation of a **REST API**.
- **Plain text**: sometimes the simplest possible view is just raw text, useful for logging, debugging, or feeding into command‑line tools.

The choice of output format depends entirely on the **consumer** of the View. The Model and Controller remain unchanged; only the View layer varies.

---


## 3. User Interface Design Principles

User Interface Design is a discipline that blends psychology, design, and engineering. While aesthetics are subjective, there are widely agreed‑upon principles that make an interface **simple** and **efficient**.

### 3.1 Core Goals: Simplicity and Efficiency
- **Simplicity**: the user should immediately understand what the interface does and how to use it. The classic “push” bar on a door is self‑explanatory; you press it and the door opens.
- **Efficiency**: the user achieves their goal with minimal effort. Keyboard shortcuts in Gmail (e.g., `c` to compose, `e` to archive) allow expert users to navigate without moving their hands to the mouse.

These two goals often reinforce each other, but they can conflict. A voice‑controlled door might be simple (“just say ‘open’”), but it’s slower and less reliable than a push bar for most people.

### 3.2 Aesthetics vs. Accessibility
- **Aesthetics**: how good something looks. It is subjective but heavily influenced by cultural trends, colour theory, and principles of visual hierarchy. The evolution of iOS icons from glossy, gradient‑filled designs to flat, minimalist icons illustrates changing aesthetic preferences. The lecture advises beginners to **rely on established design systems** (like Bootstrap, Material Design) rather than inventing their own colour schemes and layouts, because these systems have been refined by professional designers and tested on millions of users.
- **Accessibility**: how usable the interface is for people with disabilities (visual, auditory, motor, cognitive). There is often a tension: high‑contrast, large‑font designs needed for low‑vision users may look “loud” to fully‑sighted users. The principle is that **accessibility is not optional**; it is a requirement for ethical, inclusive, and often legally compliant software.

### 3.3 The Systematic Process of UI Design
Building a good UI is not guesswork. It follows an engineering process:

1. **Functionality Requirements Gathering**: Talk to stakeholders (the client, the actual users) to understand what the application must do. For a gradebook, the academic section might need:
   - Bulk import of students from a CSV file.
   - Manual entry of individual marks.
   - A way to lock results after finalisation.
   - A dashboard showing failure rates per course.

2. **User and Task Analysis**: Identify who will use the system and what tasks they will perform. The administrator has different needs than a student viewing their own grades. Task analysis breaks down each task into steps, revealing where the UI must provide guidance or shortcuts.

3. **Prototyping**: Create low‑fidelity **wireframes** or **mock‑ups**. These show layout, navigation, and key elements without full functionality. They are cheap to produce and iterate. Feedback from users at this stage prevents costly rework later.

4. **Implementation**: Translate the approved prototype into actual code (HTML, CSS, server‑side logic).

5. **User Acceptance Testing**: Real users perform tasks using the implemented application. Observations and feedback may lead to further refinements. This cycle often repeats (agile development).

---

## 4. Usability Heuristics (Jakob Nielsen’s 10 Heuristics)

In 1994, Jakob Nielsen published a set of **ten general principles** for interaction design. They are called *heuristics* because they are rules of thumb, not rigid laws. They apply to virtually any user interface, from a website to a microwave oven.

1. **Visibility of system status**  
   The system should always keep users informed about what is going on, through appropriate feedback within reasonable time.  
   *Example*: The mouse cursor changes to a hand over a hyperlink, indicating it is clickable. A progress bar shows how much of a file upload remains.

2. **Match between system and the real world**  
   The system should speak the user’s language, with words, phrases and concepts familiar to the user, rather than system‑oriented terms. Follow real‑world conventions, making information appear in a natural and logical order.  
   *Example*: On a stove, the knob layout mirrors the burner layout. In a shopping cart, the “checkout” process mimics a physical store.

3. **User control and freedom**  
   Users often choose system functions by mistake and need a clearly marked “emergency exit” to leave the unwanted state without having to go through an extended dialogue. Support undo and redo.  
   *Example*: “Undo Send” in Gmail; the recycle bin on desktops.

4. **Consistency and standards**  
   Users should not have to wonder whether different words, situations, or actions mean the same thing. Follow platform and industry conventions.  
   *Example*: Links are always underlined and blue by default; changing this breaks user expectations. All “Save” buttons across an application should look and behave identically.

5. **Error prevention**  
   Even better than good error messages is a careful design which prevents a problem from occurring in the first place.  
   *Example*: Drop‑down menus instead of free‑text fields for selecting a country; they limit the input to valid options. Confirmation dialogs before deleting an account.

6. **Recognition rather than recall**  
   Minimise the user’s memory load by making objects, actions, and options visible. The user should not have to remember information from one part of the dialogue to another.  
   *Example*: A toolbar with icons and tooltips is easier than requiring users to memorise keyboard shortcuts (though shortcuts can be provided for experts – see #7).

7. **Flexibility and efficiency of use**  
   Accelerators – unseen by the novice user – may often speed up the interaction for the expert user such that the system can cater to both inexperienced and experienced users. Allow users to tailor frequent actions.  
   *Example*: Keyboard shortcuts (Ctrl+S, Ctrl+Z). Macros in Excel.

8. **Aesthetic and minimalist design**  
   Dialogues should not contain information which is irrelevant or rarely needed. Every extra unit of information in a dialogue competes with the relevant units of information and diminishes their relative visibility.  
   *Example*: Google’s homepage, which consists of a logo, a search box, and two buttons. Cluttered interfaces slow down task completion.

9. **Help users recognise, diagnose, and recover from errors**  
   Error messages should be expressed in plain language (no codes), precisely indicate the problem, and constructively suggest a solution.  
   *Example*: Instead of “Error 404”, a page saying “We couldn’t find that page. Here are some links that might help.”

10. **Help and documentation**  
    Even though it is better if the system can be used without documentation, it may be necessary to provide help and documentation. Any such information should be easy to search, focused on the user’s task, list concrete steps to be carried out, and not be too large.  
    *Example*: Context‑sensitive help: clicking a “?” icon next to a field explains what that field does.

These heuristics are not step‑by‑step instructions but lenses through which to evaluate a design. The lecture strongly encourages reading the full Nielsen Norman Group articles for a deeper understanding.

---

## 5. Tools for View Design – Wireframing

Before writing any code, it is best to sketch the structure of the user interface. A **wireframe** is a visual guide that represents the skeletal framework of a page. It shows:

- **Information hierarchy**: what content is most important, where it is placed.
- **Navigation**: how users move between pages or sections.
- **Layout**: the arrangement of blocks (header, footer, sidebar, main content area).
- **Placeholder content**: typically “lorem ipsum” text, grey boxes for images.

Wireframes are deliberately devoid of colour, typography, and detailed graphics. Their purpose is to focus on **structure and functionality**, not final aesthetics.

**Example from the lecture**: A wireframe for a user directory profile might show a large placeholder for the user’s photo at the top, the name directly beneath, then contact details (email, phone), a “categories” tag cloud, a short description, and some thumbnails for attachments. The wireframe helps the client and developer agree on the essential elements before any design work.

**Tools for wireframing**:
- **Pen and paper**: the fastest, cheapest way. Use graph paper to keep proportions.
- **Dedicated software**: Lucidchart, Balsamiq, Sketch, Figma, Adobe XD. Many offer free tiers for students.
- The process is iterative: start with a low‑fidelity sketch, present it to stakeholders, incorporate feedback, then gradually increase fidelity. A wireframe can evolve into a **mock‑up** (adding colours and fonts) and finally a **prototype** (clickable with limited functionality).

---

## 6. Tools for View Implementation – Programmatic HTML and Templates

Once the wireframe is approved, we must turn it into actual HTML. There are several approaches, each with trade‑offs.

### 6.1 Direct Print Statements
The most basic method: use `print()` to output HTML strings.
```python
print("<html><head><title>My Page</title></head>")
print("<body><h1>Welcome</h1></body></html>")
```
This works for tiny examples but is unmaintainable for real applications:
- It is error‑prone (forgotten closing tags, mismatched quotes).
- The code is cluttered, making it hard to see the page structure.
- Mixing Python logic with HTML makes both harder to read.

### 6.2 Helper Functions for HTML Tags
A slight improvement: define functions that generate tags.
```python
def h1(text):
    return f"<h1>{text}</h1>"
```
This reduces typos and can include attribute handling. Libraries like **pyhtml** take this approach to the extreme, allowing us to write:
```python
from pyhtml import *
t = html(
    head(title("Test Page")),
    body(
        h1("Hello"),
        p("This is a paragraph.")
    )
)
print(t.render())
```
The output is valid, indented HTML. Because it’s pure Python, we can use loops, conditionals, and list comprehensions to build complex tables or lists. However, the resulting code is still a tree of function calls that mirrors the HTML structure. If a designer later wants to change the layout, they must wade through Python code. This violates the separation of concerns between development and design.

### 6.3 Templating Engines
A **template** is a text file that contains the static parts of the output (the “skeleton”) with special placeholders for dynamic data. The application code reads the template, provides the data, and the template engine produces the final output. This cleanly separates the **presentation logic** (in the template) from the **business logic** (in the Python code).

**Simple string templates (Python’s `string.Template`)**:
```python
from string import Template
t = Template("$name is the $job of $company.")
print(t.substitute(name="Tim Cook", job="CEO", company="Apple Inc."))
```
This is good for small substitutions but lacks features like loops and conditionals.

**Jinja2** (the template engine used with Flask):
- Uses `{{ variable }}` for expression substitution.
- Uses `{% statement %}` for control flow (loops, conditionals).
- Supports template inheritance (a base template with blocks that child templates can override).
- Auto‑escapes HTML to prevent XSS attacks by default.

A Jinja2 template for a student marks table might look like:
```html
<table>
  <tr><th>Course</th><th>Marks</th></tr>
  {% for row in marks %}
  <tr><td>{{ row.course }}</td><td>{{ row.score }}</td></tr>
  {% endfor %}
</table>
```
The Python code is minimal: load the template, call `render(marks=data)`, and return the result. The template is readable by front‑end developers who may not know Python.

The lecture emphasises that **Jinja2 can generate any text format**, not just HTML. You could generate JSON, CSV, email bodies, or even Python source code. This flexibility makes templating a universal tool in a developer’s toolkit.

---

## 7. Jinja2 Templating in Python (Screencasts)

The screencasts provide a hands‑on walkthrough of using Jinja2 to generate HTML dynamically from Python data.

### 7.1 Setup and Virtual Environment
- Create a project directory.
- Set up a **Python virtual environment** (`python3 -m venv env`). This isolates project dependencies from the system Python, preventing version conflicts.
- Activate it (`source env/bin/activate`).
- Install Jinja2 with `pip install jinja2`. The `requirements.txt` file (generated via `pip freeze > requirements.txt`) lists all dependencies so others can replicate the environment.

### 7.2 Basic Jinja2 Usage
```python
from jinja2 import Template
t = Template("Hello {{ name }}!")
print(t.render(name="Thej"))
```
Key points:
- `Template` is a class that compiles the template string into an internal representation.
- `render()` takes keyword arguments corresponding to the variables used in the template.
- The double curly braces `{{ ... }}` denote an **expression** that will be evaluated and inserted into the output.

### 7.3 Control Structures: Loops and Conditionals
Jinja2 supports `{% for %}` and `{% if %}`:
```html
<ul>
{% for item in items %}
  <li>{{ item.name }}</li>
{% endfor %}
</ul>
```
Unlike Python, Jinja2 requires an explicit `{% endfor %}` because indentation is not used to denote blocks (templates are a mix of HTML and logic). Conditionals look similar:
```html
{% if user.is_admin %}
  <a href="/admin">Admin Panel</a>
{% endif %}
```

### 7.4 Separating Template from Code
Instead of embedding the template as a string in Python, it is best to keep it in a separate file, e.g., `template.html.jinja2`. The Python code then reads the file:
```python
with open('template.html.jinja2') as file:
    template_str = file.read()
t = Template(template_str)
```
This separation means that a designer can edit the template with an HTML editor, while the developer works on the Python logic.

### 7.5 Rendering a Complete HTML Table
The example takes a list of dictionaries (Jnanpith awardees) and generates an HTML table:
- The template defines the `<table>`, `<thead>`, and a `<tbody>` with a `{% for %}` loop over the data.
- The Python data is a list of dicts: `[{'year': 1965, 'awardee': 'G. Sankara Kurup', 'language': 'Malayalam'}, ...]`.
- In the template, each row accesses dictionary keys using Python‑like syntax: `{{ awardee['year'] }}` or `{{ awardee.year }}`.
- The rendered HTML is written to a file (`janapith.html`) which can then be opened in a browser.

### 7.6 Advantages of Jinja2
- **Conciseness**: A few lines of template can replace dozens of `print` statements.
- **Readability**: The template looks almost like the final HTML, with minimal logic intruding.
- **Extensibility**: Jinja2 supports macros (reusable snippets), filters (e.g., `{{ name|title }}`), and template inheritance (a “base” template with a common layout, and “child” templates that fill in the content blocks).
- **Security**: By default, Jinja2 escapes HTML special characters in substituted variables, thwarting Cross‑Site Scripting (XSS) attacks.

The screencast encourages further exploration of Jinja2’s documentation for advanced features like template inheritance, which is essential for maintaining a consistent look across many pages of an application.

---

## 8. Accessibility in Web Applications

Accessibility (often abbreviated as **a11y**) means designing web content so that people with disabilities can perceive, understand, navigate, and interact with it. It is a fundamental aspect of ethical software development and is mandated by laws in many countries.

### 8.1 Why Accessibility Matters
- **Moral imperative**: The web should be open to everyone, regardless of ability.
- **Legal requirements**: Many governments require public websites to meet accessibility standards (e.g., Section 508 in the US, EN 301 549 in the EU).
- **Business case**: Accessible sites reach a larger audience, rank better in search engines (semantic HTML aids SEO), and generally offer a better user experience for *all* users (e.g., captions help in noisy environments).
- **Types of disabilities** include:
  - **Visual**: blindness, low vision, colour blindness.
  - **Auditory**: deafness, hard of hearing.
  - **Motor**: inability to use a mouse, slow response time, limited fine motor control.
  - **Cognitive**: learning disabilities, distractibility, difficulty reading.

### 8.2 WCAG – Web Content Accessibility Guidelines
The W3C publishes the **Web Content Accessibility Guidelines (WCAG)**. The current version (2.1) is organised around four principles, often remembered by the acronym **POUR**:

1. **Perceivable** – Information and user interface components must be presentable to users in ways they can perceive.
   - Provide **text alternatives** for non‑text content (`alt` attributes for images, captions for videos).
   - Content must be adaptable and distinguishable (e.g., sufficient colour contrast, resizable text without loss of content).

2. **Operable** – User interface components and navigation must be operable.
   - All functionality must be available from a **keyboard** (no mouse‑only interactions).
   - Users must have enough time to read and use content (no auto‑advancing carousels without pause controls).
   - Do not design content in a way that is known to cause **seizures** (no flashing content > 3 times per second).
   - Provide ways to help users navigate, find content, and determine where they are (skip‑to‑main‑content links, meaningful page titles).

3. **Understandable** – Information and the operation of the user interface must be understandable.
   - Text content must be readable and understandable (use plain language).
   - Web pages must appear and operate in predictable ways (consistent navigation, no unexpected pop‑ups).
   - Help users avoid and correct mistakes (input validation, clear error messages).

4. **Robust** – Content must be robust enough that it can be interpreted reliably by a wide variety of user agents, including assistive technologies.
   - Use valid, semantic HTML so that screen readers and other tools can parse the document correctly.
   - ARIA (Accessible Rich Internet Applications) attributes can enhance accessibility of dynamic content, but should be used as a supplement, not a substitute for proper HTML.

### 8.3 Practical Accessibility Techniques
- **Semantic HTML**: Use `<header>`, `<nav>`, `<main>`, `<article>`, `<aside>`, `<footer>` instead of generic `<div>`s. This provides a logical structure that screen readers can navigate.
- **Form labels**: Every form input should have a `<label>` associated with it, so screen readers announce what each field is for.
- **Focus management**: Ensure that keyboard focus order follows the visual order. Do not trap focus in a modal without a way to close it via keyboard.
- **Colour and contrast**: Ensure a contrast ratio of at least 4.5:1 for normal text. Do not rely on colour alone to convey information (e.g., add an icon next to error messages, not just a red border).
- **Testing**: Use automated tools (WAVE, axe, Lighthouse) and manual testing (keyboard‑only navigation, screen reader testing with NVDA or VoiceOver). The W3C provides a full suite of techniques for meeting each WCAG success criterion.

The lecture strongly stresses that **accessibility is an integral part of development, not an afterthought**. Designing with accessibility in mind from the beginning is far cheaper and results in a better product than retrofitting later.

---

## 9. Command Line Arguments in Python

This screencast covers a fundamental skill: executing Python scripts from the terminal and passing parameters to them. This is crucial for building server‑side scripts that can be configured without modifying source code.

### 9.1 Running a Python Script from the Terminal
Normally, an IDE provides a “Run” button. On a server or in an automated pipeline, we use the command line:
```bash
python script.py
```
(On some systems, `python3` is required to avoid invoking Python 2.)

If the script contains only `print("Hello")`, that output appears in the terminal. The script can also use `input()` to accept user input interactively.

### 9.2 Passing Command Line Arguments
Arguments are written after the script name, separated by spaces:
```bash
python hello.py 123 Abhishek
```
These arguments are accessible inside the script via the **`sys.argv`** list, provided by the built‑in `sys` module.

### 9.3 The `sys.argv` List
- `sys.argv` is a Python list of strings.
- `sys.argv[0]` is **always the script name** (e.g., `hello.py`).
- `sys.argv[1]` is the first argument, `sys.argv[2]` the second, and so on.
- The length `len(sys.argv)` includes the script name, so if you pass two arguments, the length is 3.

Example:
```python
import sys
print(f"Total parameters: {len(sys.argv)}")
print(f"First parameter: {sys.argv[1]}")
print(f"Second parameter: {sys.argv[2]}")
```
Running `python hello.py 123 Abhishek` prints:
```
Total parameters: 3
First parameter: 123
Second parameter: Abhishek
```

### 9.4 Important Notes
- **All arguments are strings**, even if they look like numbers. You must explicitly convert them using `int()`, `float()`, etc., if you need numerical values.
- You can pass any number of arguments; the script can ignore excess ones or use only what it needs.
- Command line arguments are a simple way to parameterise scripts (e.g., specify a port number, a filename, or a logging level). For more complex option parsing, Python’s `argparse` module is recommended, but `sys.argv` suffices for basic cases.

This technique is the foundation for writing server‑side utilities, background jobs, and even for the first steps towards building an HTTP server that responds to different paths.

---

## 10. Browser Developer Tools

Modern browsers are not just for viewing web pages; they include a powerful suite of **developer tools** (DevTools) for inspecting, debugging, and profiling web applications. The screencast explores the most essential panels using Chrome as an example, but Firefox’s tools are nearly identical.

### 10.1 Opening DevTools
- Right‑click on any element and select **Inspect**, or press `F12` (or `Ctrl+Shift+I`).
- The DevTools pane can be docked to the bottom, side, or popped out as a separate window.

### 10.2 The Elements (Inspector) Panel
- Shows the live DOM tree of the current page. As you hover over nodes, the corresponding area on the page is highlighted.
- You can edit the HTML directly: double‑click a tag, modify text, or add/delete elements. These changes are temporary and vanish on refresh.
- The **Styles** sub‑panel on the right shows all CSS rules that apply to the selected element, grouped by selector. The **Computed** sub‑panel shows the final resolved values.
- You can toggle individual rules on/off, edit property values, or add new styles on the fly. This is invaluable for experimenting with design without touching source code.
- The cascade is visible: overridden properties are shown with a strikethrough, and clicking the arrow next to a property reveals its origin.

### 10.3 The Console Panel
- A JavaScript REPL (Read‑Eval‑Print Loop). You can type arbitrary JavaScript and see the results immediately.
- It is the primary output destination for `console.log()`, `console.error()`, `console.warn()`, etc.
- Useful for debugging: instead of intrusive `alert()` calls, developers log messages, objects, and variables to the console.
- `console.table(data)` prints an array or object as a formatted table, making it easier to inspect structured data.
- `console.time("label")` and `console.timeEnd("label")` measure the execution time of code blocks.
- `console.clear()` wipes the console.

### 10.4 The Sources Panel
- Shows all files that make up the current page: HTML, CSS, JavaScript, images, fonts.
- For JavaScript, you can **set breakpoints** by clicking on line numbers. When execution reaches that line, the debugger pauses, allowing you to inspect variables, step through code, and watch the call stack.
- This is the same debugging paradigm found in IDE debuggers and is essential for understanding complex client‑side logic.

### 10.5 The Network Panel
- Records every HTTP request the page makes (HTML, CSS, JS, images, API calls).
- For each request, it shows:
  - **Status code**, type (`GET`, `POST`), size, and **time**.
  - **Headers** (both request and response) – useful for checking cookies, content types, caching directives.
  - **Preview/Response** – see the actual data returned.
- The **waterfall** visualisation breaks down the timing of each request (queuing, DNS lookup, TCP connection, SSL handshake, waiting for server, downloading content). This helps identify bottlenecks.
- You can filter requests by type (XHR for API calls, JS, CSS, etc.).
- **Preserve log** keeps requests visible across page navigations.
- **Throttling** allows simulating slow network conditions (e.g., Slow 3G) to test performance on mobile.

### 10.6 The Application Panel
- Inspects client‑side storage:
  - **Local Storage** and **Session Storage**: key‑value stores accessible via JavaScript. You can view, edit, and delete entries.
  - **Cookies**: see all cookies set by the page, their domains, paths, expiry dates, and flags (`HttpOnly`, `Secure`).
  - **Cache Storage**: view cached responses (e.g., service worker caches).
  - **IndexedDB**: a low‑level API for storing large amounts of structured data. The panel allows browsing databases, object stores, and records.

### 10.7 Other Panels
- **Performance**: record and analyse runtime performance (CPU usage, rendering, scripting).
- **Memory**: profile memory usage and find leaks.
- **Lighthouse**: an integrated audit tool that scores the page on performance, accessibility, best practices, and SEO.

The screencast emphasises that **the browser is the developer’s most important tool**. Learning DevTools thoroughly pays enormous dividends in debugging productivity. The examples with `httpbin` – a service for testing HTTP requests – show how to examine POST form data, verify response headers, and understand the whole request‑response cycle.

---

## Summary of Week 3

- **MVC** provides a proven structure: Model (data), View (presentation), Controller (logic). This separation is the backbone of the course’s approach to app development.
- **Views** are any output presented to a user or another machine. They must adapt to diverse hardware, can be static or dynamic, and may produce HTML, images, JSON, or plain text.
- **UI Design** aims for simplicity and efficiency. It follows a systematic process of requirements, prototyping, and testing. Established heuristics (Nielsen’s 10) guide designers towards usable products.
- **Wireframes** are low‑fidelity blueprints that define layout and navigation before coding begins.
- **HTML generation** can be done programmatically (using helper functions or libraries like pyhtml) but **templates** (especially Jinja2) offer a superior separation of concerns. Jinja2 is a powerful, extensible templating engine that supports variables, loops, conditionals, and inheritance.
- **Accessibility** is a core requirement, not a nice‑to‑have. The WCAG’s POUR principles (Perceivable, Operable, Understandable, Robust) provide a comprehensive framework. Semantic HTML, keyboard navigation, contrast, and text alternatives are key techniques.
- **Command line arguments** enable scripts to accept parameters at runtime via `sys.argv`, a crucial skill for backend development.
- **Browser DevTools** are the developer’s Swiss Army knife: Elements for HTML/CSS debugging, Console for JavaScript logging, Sources for debugging, Network for request inspection, and Application for storage inspection.

This week moves from abstract design principles to concrete tools and techniques, equipping students to begin building the **View** layer of their web applications with modern, maintainable, and inclusive practices.